   

# ML: prever titulares de crédito com alta probabilidade de inadimplência

Uma vez que todos os dados estão carregados e protegidos (a parte de **unificação de dados**), podemos prosseguir para explorar, entender e usar os dados para criar insights acionáveis - **decisão baseada em dados**.


Construiremos modelos de machine learning (ML) para impulsionar três resultados de negócio:
1. Identificar clientes atualmente sub-bancarizados com alta capacidade creditícia para que possamos oferecer instrumentos de crédito,
2. Prever titulares de crédito atuais com alta probabilidade de inadimplência junto com a perda dada a inadimplência, e
3. Oferecer micro-empréstimos instantâneos (Compre Agora, Pague Depois) quando um cliente não possui o limite de crédito ou saldo em conta necessário para concluir uma transação.

In [0]:
%pip install databricks-sdk==0.36.0 mlflow==2.19.0 databricks-feature-store==0.17.0
dbutils.library.restartPython()

In [0]:
from datetime import date
import plotly.express as px
from pyspark.sql.functions import col
from pyspark.sql import functions as F

# Preencha apenas estas variáveis
CATALOGO = ""
SCHEMA = ""
PREFIXO = ""

identificador_schema = f"{CATALOGO}.{SCHEMA}"

           

## Exploração de dados e criação de Features

<br/><br/>
O primeiro passo como Cientista de Dados é explorar nossos dados e entendê-los para criar Features.

<br/>

É aqui que usamos nossas tabelas existentes e transformamos os dados para que estejam prontos para nossos modelos de ML. Essas features serão posteriormente armazenadas no Databricks Feature Store (veja abaixo) e usadas para treinar os modelos de ML mencionados anteriormente.

<br/>

Vamos começar com alguma exploração de dados. O Databricks vem com Profiling de Dados integrado para ajudá-lo a iniciar esse processo.

In [0]:
display(spark.sql(f"SELECT * FROM {identificador_schema}.{PREFIXO}customer_gold WHERE tenure_months BETWEEN 10 AND 150"))

In [0]:
data = spark.table(f"{identificador_schema}.{PREFIXO}customer_gold") \
              .where("tenure_months BETWEEN 10 AND 150") \
              .groupBy("tenure_months", "education").sum("income_monthly") \
              .orderBy('education').toPandas()

px.bar(data, x="tenure_months", y="sum(income_monthly)", color="education", title="Wide-Form Input")

   

# Construindo nossas Features para Riscos de Inadimplência de Crédito

Para construir nosso modelo que prevê riscos de inadimplência de crédito, precisaremos de várias features. Para melhorar nossa governança e centralizar nossos dados para múltiplos projetos de ML, podemos salvar nossas features de ML usando uma Feature Store.

In [0]:
customer_gold_features = (spark.table(f"{identificador_schema}.{PREFIXO}customer_gold")
                               .withColumn('age', int(date.today().year) - col('birth_year'))
                               .select('cust_id', 'education', 'marital_status', 'months_current_address', 'months_employment', 'is_resident',
                                       'tenure_months', 'product_cnt', 'tot_rel_bal', 'revenue_tot', 'revenue_12m', 'income_annual', 'tot_assets', 
                                       'overdraft_balance_amount', 'overdraft_number', 'total_deposits_number', 'total_deposits_amount', 'total_equity_amount', 
                                       'total_UT', 'customer_revenue', 'age', 'avg_balance', 'num_accs', 'balance_usd', 'available_balance_usd')).dropDuplicates(['cust_id'])
display(customer_gold_features)

In [0]:
telco_gold_features = (spark.table(f"{identificador_schema}.{PREFIXO}telco_gold")
                            .select('cust_id', 'is_pre_paid', 'number_payment_delays_last12mo', 'pct_increase_annual_number_of_delays_last_3_year', 'phone_bill_amt', \
                                    'avg_phone_bill_amt_lst12mo')).dropDuplicates(['cust_id'])
display(telco_gold_features)

In [0]:
fund_trans_gold_features = spark.table(f"{identificador_schema}.{PREFIXO}fund_trans_gold").dropDuplicates(['cust_id'])

for c in ['12m', '6m', '3m']:
  fund_trans_gold_features = fund_trans_gold_features.withColumn('tot_txn_cnt_'+c, col('sent_txn_cnt_'+c)+col('rcvd_txn_cnt_'+c))\
                                                     .withColumn('tot_txn_amt_'+c, col('sent_txn_amt_'+c)+col('rcvd_txn_amt_'+c))

fund_trans_gold_features = fund_trans_gold_features.withColumn('ratio_txn_amt_3m_12m', F.when(col('tot_txn_amt_12m')==0, 0).otherwise(col('tot_txn_amt_3m')/col('tot_txn_amt_12m')))\
                                                   .withColumn('ratio_txn_amt_6m_12m', F.when(col('tot_txn_amt_12m')==0, 0).otherwise(col('tot_txn_amt_6m')/col('tot_txn_amt_12m')))\
                                                   .na.fill(0)
display(fund_trans_gold_features)

In [0]:
feature_df = customer_gold_features.join(telco_gold_features.alias('telco'), "cust_id", how="left")
feature_df = feature_df.join(fund_trans_gold_features, "cust_id", how="left")
display(feature_df)

           

# Databricks Feature Store


Uma vez que nossas features estão prontas, vamos salvá-las no Databricks Feature Store.

Por baixo dos panos, a Feature Store é apoiada por uma tabela Delta Lake. Isso permitirá a descoberta e reutilização de nossas features em toda a organização, aumentando a eficiência da equipe.


O Databricks Feature Store traz capacidades avançadas para acelerar e simplificar sua jornada de ML, como suporte a point-in-time e online-store, buscando suas features em milissegundos para Serving em tempo real.

### Por que usar o Databricks Feature Store?

O Databricks Feature Store é totalmente integrado com outros componentes do Databricks.

* **Descoberta**. A interface do Feature Store, acessível a partir do workspace Databricks, permite que você navegue e pesquise features existentes.

* **Linhagem**. Quando você cria uma tabela de features com o Feature Store, as fontes de dados usadas para criar a tabela são salvas e acessíveis. Para cada feature em uma tabela, você também pode acessar os modelos, notebooks, jobs e endpoints que usam a feature.

* **Busca de features em Batch e Online para serving em tempo real**. Quando você usa features do Feature Store para treinar um modelo, o modelo é empacotado com metadados das features. Quando você usa o modelo para scoring em batch ou inferência online, ele automaticamente recupera as features do Feature Store. O chamador não precisa conhecê-las ou incluir lógica para buscar ou juntar features para pontuar novos dados. Isso torna a implantação e atualização de modelos muito mais fácil.

* **Buscas point-in-time**. O Feature Store suporta séries temporais e casos de uso baseados em eventos que requerem correção temporal (point-in-time).


Para mais detalhes sobre o Databricks Feature Store, execute `dbdemos.install('feature-store')`

In [0]:
from databricks import feature_store
fs = feature_store.FeatureStoreClient()

feature_table_name = f"{identificador_schema}.{PREFIXO}credit_decisioning_features_only_example"
  
fs.create_table(
    name=feature_table_name,
    primary_keys=["cust_id"],
    df=feature_df,
    description="Features for Credit Decisioning.")